# Stage 4 — Pretraining

**Goal:** turn random weights into a model that writes English, in about 45
minutes on a free T4.

The training loop itself is ordinary. The things worth your attention are
operational, and each one is a lesson that transfers to real training runs.

### fp16, not bf16 — and why that forces a GradScaler

A Colab T4 is Turing (compute capability 7.5). Turing has **no bf16 support**, so
we use fp16 autocast. fp16 has a much narrower exponent range, and small gradients
simply **underflow to zero** — training silently stalls.

The fix is loss scaling: multiply the loss by a large factor before `.backward()`
so gradients land inside fp16's representable range, then divide it back out
before the optimizer step. `GradScaler` does this, and adapts the factor
automatically when it detects overflow.

On an A100 or newer you'd use bf16 and delete the scaler entirely — bf16 has the
same exponent range as fp32 and doesn't need the trick.

### Durable checkpoints

Free-tier Colab reclaims runtimes without warning. **A checkpoint on the
runtime's local disk is not a checkpoint.** State is pushed to a Hub repo every
500 steps.

Resume restores the optimizer moments, the scaler's scale factor, *and the data
sampler's RNG state*. Restoring only the weights — which is what most tutorial
code does — silently restarts the data order, so the model re-reads tokens it has
already seen while you believe it is making fresh progress.

### Gradient accumulation and the order of operations

The effective batch is 65,536 tokens, which won't fit in 16 GB at once, so it's
split into 4 micro-batches. Two details that are easy to get wrong:

1. Each micro-batch's loss is divided by `grad_accum_steps` **before** backward,
   so the accumulated gradient equals the gradient of the mean loss.
2. `scaler.unscale_()` must be called **before** clipping. Clip an
   unscaled-by-the-loss-scale gradient and the threshold is meaningless — you'd
   be clipping numbers that are ~65,536× too large, so clipping never triggers.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Set this to YOUR GitHub repo once; every notebook uses the same cell.
REPO_URL = "https://github.com/pythonstudentiam/e2e_llm_demo.git"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/e2e_llm_demo")
WORK = Path("/content/work")          # scratch: data + checkpoints (ephemeral!)
WORK.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))

# Colab ships torch; these are the rest. -q to keep the log readable.
%pip install -q sentencepiece "datasets>=3.0" "transformers>=4.45" "huggingface_hub>=0.30"

# HF token from the Colab Secrets panel (key icon, left sidebar). Name it HF_TOKEN
# and enable notebook access. Never paste a token into a cell.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception as e:
    print(f"No HF_TOKEN yet ({e}). Needed from notebook 04 onward.")

import torch
print(f"torch {torch.__version__} | CUDA {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# A T4 is expected. Anything without CUDA means Runtime > Change runtime type > T4 GPU.
import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > Hardware accelerator: T4 GPU, then re-run."
)
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} | compute capability {cap[0]}.{cap[1]} | {mem:.1f} GB")

# Turing (7.5) has no bf16. That is why training uses fp16 + GradScaler.
if cap[0] < 8:
    print("\n-> Pre-Ampere GPU: bf16 unavailable, fp16 autocast + loss scaling it is.")
else:
    print("\n-> Ampere or newer: bf16 would work here and would let you drop the GradScaler.")

In [ ]:
from tinyllm import config
from tinyllm.config import (
    model_cfg, train_cfg, data_cfg, tok_cfg, sft_cfg, gen_cfg, quant_cfg, serve_cfg, hub,
)

print(config.summary())

## 4.1 — Make sure the data is here

`/content/work` does not survive a runtime reset. If `train.bin` is gone, this
rebuilds it from the tokenizer on the Hub — a few minutes, no need to re-run
earlier notebooks.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
from tinyllm.tokenizer import load_sp
from tinyllm.data import load_tokens, prepare_corpus, describe

data_dir = WORK / "data"
tok_dir = WORK / "tokenizer"
sp_path = tok_dir / "tokenizer.model"

# Tokenizer: local if present, otherwise pull the copy stage 2 pushed.
if not sp_path.exists():
    print("fetching tokenizer from the Hub...")
    tok_dir.mkdir(parents=True, exist_ok=True)
    got = hf_hub_download(repo_id=hub.ckpt_repo, filename="tokenizer/tokenizer.model")
    sp_path.write_bytes(Path(got).read_bytes())

sp = load_sp(sp_path)
print(f"tokenizer: {sp.GetPieceSize():,} pieces")

# Token files. Three tiers, cheapest first:
#   1. already on this runtime          (free)
#   2. on the Hub from notebook 2.7     (~1 min)
#   3. repack from the raw corpus       (~15 min)
if not (data_dir / "train.bin").exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    pulled = True
    for name in ["train.bin", "train.json", "val.bin", "val.json"]:
        try:
            got = hf_hub_download(repo_id=hub.ckpt_repo, filename=f"data/{name}")
            (data_dir / name).write_bytes(Path(got).read_bytes())
            print(f"pulled {name} from the Hub")
        except Exception as e:
            print(f"could not pull {name} ({type(e).__name__})")
            pulled = False
            break

    if not pulled:
        print("\nfalling back to repacking the corpus (~10-15 min)")
        prepare_corpus(sp, data_dir)

train_tokens = load_tokens(data_dir / "train.bin")
val_tokens = load_tokens(data_dir / "val.bin")
print(f"\ntrain: {len(train_tokens):,} tokens")
print(f"val:   {len(val_tokens):,} tokens")

## 4.2 — What a training sample actually looks like

Worth seeing once. `x` and `y` are the same window offset by one: at every
position the model predicts the next token, so a single forward pass produces
`seq_len` predictions rather than one. That is what makes causal LM training
efficient, and it's why the loss is an average over 512 predictions per sequence.

In [ ]:
from tinyllm.data import get_batch
import numpy as np

x, y = get_batch(train_tokens, batch_size=2, seq_len=data_cfg.seq_len,
                 device="cuda", rng=np.random.default_rng(0))
print(f"x {tuple(x.shape)}  y {tuple(y.shape)}  dtype {x.dtype}")
print(f"\ny is x shifted by one: {bool((x[0, 1:] == y[0, :-1]).all())}")

print("\nfirst 40 tokens of sample 0, decoded:")
print(" ", repr(sp.DecodeIds(x[0, :40].tolist())))

print("\nthe model's job at each position:")
for i in range(5):
    print(f"  given ...{sp.IdToPiece(int(x[0, i]))!r:<14} -> predict {sp.IdToPiece(int(y[0, i]))!r}")

## 4.3 — The learning-rate schedule

Linear warmup then cosine decay. Warmup exists because Adam's second-moment
estimate is garbage for the first few dozen steps — stepping at full LR before it
stabilizes is a common way to blow up a run in its first minute.

In [ ]:
import matplotlib.pyplot as plt

steps = list(range(train_cfg.max_steps))
lrs = [train_cfg.lr_at(s) for s in steps]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(steps, lrs, color="#4C78A8", linewidth=2)
ax.axvline(train_cfg.warmup_steps, color="#E45756", linestyle="--",
           label=f"warmup ends ({train_cfg.warmup_steps})")
ax.axhline(train_cfg.min_lr, color="#54A24B", linestyle=":",
           label=f"floor ({train_cfg.min_lr:.1e})")
ax.set_xlabel("step"); ax.set_ylabel("learning rate")
ax.set_title("Warmup + cosine decay")
ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"peak {train_cfg.learning_rate:.2e} at step {train_cfg.warmup_steps}, "
      f"decaying to {train_cfg.min_lr:.2e}")

## 4.4 — Smoke run

**Do not skip this.** 50 steps, about a minute, exercising every code path the
real run uses: accumulation, autocast, the scaler, unscale-then-clip, evaluation,
sampling, and checkpoint writing.

It calls the same `train()` function with a smaller step budget rather than a
simplified copy — a rehearsal that runs different code proves nothing.

Finding a bug here costs a minute. Finding it 40 minutes into the real run costs
40 minutes.

In [ ]:
from tinyllm.train import smoke_test

smoke_model, smoke_hist = smoke_test(sp, train_tokens, val_tokens,
                                     out_dir=WORK / "checkpoints/smoke")

In [ ]:
# Free the smoke model before the real run -- 16 GB goes quickly otherwise.
import torch, gc
del smoke_model
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4.5 — Verify resume actually works

The smoke run wrote a checkpoint. Before trusting checkpointing with 45 minutes
of work, prove that loading one restores the model *exactly* — not approximately.

This is the check that catches the "we only saved the weights" bug, which looks
fine until you resume and the loss jumps.

In [ ]:
import torch
from tinyllm.train import build_model, make_optimizer, load_checkpoint
from tinyllm.data import iter_eval_batches

ckpt = WORK / "checkpoints/smoke/latest.pt"
assert ckpt.exists(), "smoke run did not write a checkpoint"

fresh = build_model(model_cfg, "cuda")
opt, _ = make_optimizer(fresh)
scaler = torch.amp.GradScaler("cuda")
rng = np.random.default_rng(train_cfg.seed)

step, meta = load_checkpoint(ckpt, fresh, opt, scaler, rng)
print(f"restored to step {step}")

# Same fixed batch through the restored model must give the restored loss.
fresh.eval()
xb, yb = next(iter(iter_eval_batches(val_tokens, 8, data_cfg.seq_len, 1)))
with torch.no_grad():
    loss = fresh(input_ids=xb, labels=yb).loss.item()
print(f"restored model val loss on a fixed batch: {loss:.4f}")

assert opt.state_dict()["state"], "optimizer moments were NOT restored -- resume would restart Adam"
print("optimizer moments restored: OK")
print(f"checkpoint size: {ckpt.stat().st_size / 1e6:.0f} MB "
      "(weights + Adam's two moments per parameter)")

del fresh, opt
gc.collect(); torch.cuda.empty_cache()

Note the checkpoint size — roughly 3× the model. Adam stores two moment tensors
per parameter, so optimizer state dominates. That is why the Hub push cadence is
every 500 steps rather than every 50.

## 4.6 — The real run

~5,000 steps, ~328M tokens, **roughly 45 minutes**.

While it runs, watch three things:

- **`loss`** should fall fast to ~4, then grind slowly downward. Starting value
  should be near `ln(8192) ≈ 9.01`.
- **`gn` (gradient norm)** should settle to a stable range. Repeated spikes into
  the tens mean the LR is too high.
- **`mfu`** is the fraction of the T4's peak FLOPs you're using. 15–30% is normal
  for a model this small; the kernels are too small to saturate the GPU.

Occasional fp16 loss-scale warnings from the scaler are **expected** — that's the
scaler discovering the right scale factor, not a failure.

**If the runtime disconnects:** just re-run this cell. It pulls the last
checkpoint from the Hub and continues from that step.

In [ ]:
from tinyllm.train import train

# Resume from the last checkpoint (local, else the Hub) if there is one.
#
# Safe to leave True: checkpoints carry a version, and any written by code that
# has since been fixed in a way that changes what the model learns is rejected
# and deleted rather than loaded. You will see "IGNORING existing checkpoint"
# if that happens. Set False only to force a fresh start for another reason.
RESUME = True

model, history = train(
    train_tokens, val_tokens, sp=sp,
    out_dir=WORK / "checkpoints",
    resume=RESUME,
    push_to_hub=True,
)

## 4.7 — The loss curve

Save this plot. It's the clearest single artifact of the run.

In [ ]:
import csv

rows = list(csv.DictReader((WORK / "checkpoints/metrics.csv").open()))
tr = [(int(r["step"]), float(r["train_loss"])) for r in rows if r["train_loss"]]
va = [(int(r["step"]), float(r["val_loss"])) for r in rows if r["val_loss"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(*zip(*tr), color="#4C78A8", alpha=0.5, linewidth=1, label="train")
if va:
    ax.plot(*zip(*va), color="#E45756", marker="o", markersize=3, linewidth=2, label="val")
ax.axhline(np.log(model_cfg.vocab_size), color="gray", linestyle=":",
           label=f"random ({np.log(model_cfg.vocab_size):.2f})")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy loss")
ax.set_title("Loss"); ax.legend()

ax = axes[1]
ax.plot(*zip(*tr), color="#4C78A8", alpha=0.5, linewidth=1)
if va:
    ax.plot(*zip(*va), color="#E45756", marker="o", markersize=3, linewidth=2)
ax.set_xscale("log"); ax.set_xlabel("step (log)"); ax.set_ylabel("loss")
ax.set_title("Loss, log-x — early progress is easier to read here")

ax = axes[2]
mfu = [(int(r["step"]), float(r["mfu"]) * 100) for r in rows if r["mfu"]]
if mfu:
    ax.plot(*zip(*mfu), color="#54A24B", linewidth=1)
    ax.set_xlabel("step"); ax.set_ylabel("MFU (%)")
    ax.set_title(f"Model FLOPs utilization (median {np.median([m for _, m in mfu]):.1f}%)")

for a in axes:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

print(f"first val loss: {va[0][1]:.4f}  ->  final: {va[-1][1]:.4f}")
print(f"final perplexity: {np.exp(va[-1][1]):.2f}")

## 4.8 — Prediction vs measurement

Stage 3 predicted throughput from a FLOPs count. Here is the real number.

In [ ]:
tps = [float(r["tokens_per_sec"]) for r in rows if r["tokens_per_sec"]]
median_tps = float(np.median(tps))
achieved = model_cfg.flops_per_token() * median_tps

print(f"  measured throughput   {median_tps / 1e3:.1f}k tokens/sec")
print(f"  achieved FLOPs        {achieved:.3e}")
print(f"  T4 fp16 peak          6.500e+13")
print(f"  MFU                   {achieved / 65e12:.1%}")
print()
print(f"  total elapsed         {float(rows[-1]['elapsed_s']) / 60:.1f} min")
print(f"  tokens processed      {train_cfg.total_tokens:,}")

## 4.9 — Read what it writes

The loss number is abstract. This is not.

In [ ]:
from tinyllm.evaluate import generate

for prompt in gen_cfg.eval_prompts:
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {generate(model, sp, prompt, max_new_tokens=150, seed=0)}")
    print("-" * 78)

It should be writing fluent, grammatical children's stories with a beginning and
an end. Plots may wander and characters may lose track of themselves — that is
what 15.7M parameters buys, and it's a genuinely surprising amount.

## Stage 4 gate

- [x] Smoke run exercised every code path before the real run
- [x] Resume verified to restore optimizer moments, not just weights
- [x] Validation loss well below `ln(V) = 9.01`
- [x] Checkpoints on the Hub, so this is safe from a runtime reset
- [x] Output is recognizably English

**Next:** `05_eval.ipynb`